#### Data set cleaning

Dataset: 
 
 - _music_project_en.csv_ 
 - _music_location.csv
 - _music_location_datetime.csv_

Author: Luis Sergio Pastrana Lemus  
Date: 2025-04-23

# Data Cleaning – Music Activity Dataset

### 💻 __1. Libraries__

In [18]:
from IPython.display import display, HTML
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import os
from pathlib import Path
import sys

# Define project root dynamically, gets the current directory from whick the notebook belongs and moves one level upper
project_root = Path.cwd().parent

# Add src to sys.path if it is not already
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Import function directly (more controlled than import *)
from src import *

## __2. Path to Data file__

In [ ]:
# Build route to data file and upload
data_file_path = project_root / "data" / "processed"
df_music_clean = load_dataset_from_csv(data_file_path, "music_clean.csv", sep=',', header='infer', keep_default_na=False)


In [12]:
df_music_clean = cast_datatypes(df_music_clean, 'datetime', date_format="%H:%M:%S", c_include=['time'])
df_music_clean['time'] = df_music_clean['time'].apply(lambda x: x.hour if pd.notnull(x) else None)
df_music_clean

,userid,track,artist,genre,city,time,day
0,FFB692EC,kamigata_to_boots,the_mass_missile,rock,shelbyville,20,wednesday
1,55204538,delayed_because_of_accident,andreas_ronnberg,rock,springfield,14,friday
2,20EC38,funiculi_funicula,mario_lanza,pop,shelbyville,20,wednesday
3,A3DD03C9,dragons_in_the_sunset,fire_ice,folk,shelbyville,8,monday
4,E2DC1FAE,soul_people,space_echo,dance,springfield,8,monday
...,...,...,...,...,...,...,...
59957,729CBB09,my_name,mclean,rnb,springfield,13,wednesday
59958,D08D4A55,maybe_one_day_feat_black_spade,blu_exile,hiphop,shelbyville,10,monday
59959,C5E3A0D5,jalopiina,unknown,industrial,springfield,20,friday
59960,321D0506,freight_train,chas_mcdevitt,rock,springfield,21,friday


## 🛠️ __3. Functions__

## 📊 __4. Dashboard__

In [34]:
# Initialize the app
external_stylesheets = ['https://stackpath.bootstrapcdn.com/bootstrap/4.5.2/css/bootstrap.min.css']
app = dash.Dash(__name__, external_stylesheets=external_stylesheets, compress=False)

# OPTIONS 
cities = df_music_clean["city"].unique().tolist()
days = sorted(df_music_clean["day"].unique())

# === LAYOUT ===
app.layout = html.Div(
    [
        html.H2(
            "Shelbyville vs Springfield Music Activity",
            style={"textAlign": "center", "marginTop": "10px", "marginBottom": "10px"},
        ),

        # Controles
        html.Div(
            [
                html.Div(
                    [
                        html.Label("Select Day:"),
                        dcc.Dropdown(
                            options=[{"label": day, "value": day} for day in days],
                            id="day_filter",
                            placeholder="All Days",
                            multi=True,
                        ),
                    ],
                    style={"display": "inline-block", "width": "48%", "verticalAlign": "top"},
                ),
                html.Div(
                    [
                    
                    ],
                    style={"display": "inline-block", "width": "12%"},
                ),
                html.Div(
                    [
                        html.Label("Select City:"),
                        dcc.RadioItems(
                            options=[{"label": "all", "value": "all"}]
                            + [{"label": city, "value": city} for city in cities],
                            id="city_filter",
                            value="all",
                            inline=True,
                            labelStyle={"display": "inline-block",
                                        "marginRight": "20px",   # space between each radio+label
                                        "paddingLeft": "6px",    # space between circle and text
                            },
                        ),
                    ],
                    style={"display": "inline-block", "width": "30%", "textAlign": "left"},
                ),
            ],
            style={"width": "90%", "margin": "0 auto 20px auto", "alignItems": "center"},
        ),

        # Main content: two columns (left 40% - right 60%)
        html.Div(
            [
                # Left Column: pie  + artist + track (stacked)
                html.Div(
                    [
                        html.Div(dcc.Graph(id="city_pie"), style={"height": "420px", "marginBottom": "18px"}),
                        html.Div(dcc.Graph(id="genre_bar"), style={"height": "260px", "marginBottom": "18px"}),
                        html.Div(dcc.Graph(id="artist_bar"), style={"height": "260px", "marginBottom": "18px"}),
                        html.Div(dcc.Graph(id="track_bar"), style={"height": "260px"}),
                    ],
                    style={"flex": "0 0 40%", "paddingRight": "18px"},
                ),

                # Right Column: genre + city_track_genre_artist + timeline
                html.Div(
                    [
                        # Three small ones (city_genre, city_artist, city_track) stacked
                        html.Div(
                            [
                                html.Div(dcc.Graph(id="city_genre_bar"), style={"height": "320px", "marginBottom": "8px"}),
                                html.Div(dcc.Graph(id="city_artist_bar"), style={"height": "320px", "marginBottom": "8px"}),
                                html.Div(dcc.Graph(id="city_track_bar"), style={"height": "320px"}),
                            ],
                            style={"marginBottom": "14px"},
                        ),

                        # Timeline (full width of right column, medium height)
                        html.Div(dcc.Graph(id="city_time_line"), style={"height": "300px", "marginTop": "6px"}),
                    ],
                    style={"flex": "0 0 60%", "paddingLeft": "18px"},
                ),
            ],
            style={"display": "flex", "width": "95%", "margin": "0 auto"},
        ),

        html.Div(style={"height": "30px"})  # espacio final
    ],
    style={"fontFamily": "Arial, sans-serif", "paddingBottom": "30px"},
)

# CALLBACKS 
@app.callback(
    [
        Output("city_pie", "figure"),
        Output("genre_bar", "figure"),
        Output("artist_bar", "figure"),
        Output("track_bar", "figure"),
        Output("city_genre_bar", "figure"),
        Output("city_artist_bar", "figure"),
        Output("city_track_bar", "figure"),
        Output("city_time_line", "figure"),
    ],
    [
        Input("day_filter", "value"),
        Input("city_filter", "value")
    ]
)

def update_graphs(selected_days, selected_city):
    df_filtered = df_music_clean.copy()
    grayscale = ["#969696", "#636363", "#252525"]

    # Filter by day
    if selected_days:
        df_filtered = df_filtered[df_filtered["day"].isin(selected_days)]

    if selected_city == "all":
        pass  # no filtrar, mostrar todas las ciudades
    else:
        df_filtered = df_filtered[df_filtered["city"] == selected_city]

    # We create a column "count" = 1 to be able to group and count
    df_filtered["count"] = 1

    # Pie chart - Music Activity By City
    city_pie = px.pie(
        df_filtered.groupby("city", as_index=False)["count"].count(),
        values="count", names="city",
        title="Music Activity by City",
        color_discrete_sequence=grayscale
    )

    # Music Activity By Genre
    genre_bar = px.bar(
        df_filtered.groupby("genre", as_index=False)["count"].count().sort_values("count", ascending=False),
        x="genre", y="count", title="Music Activity By Genre", color_discrete_sequence=grayscale
    )

    # Music Activity By Artist
    artist_bar = px.bar(
        df_filtered.groupby("artist", as_index=False)["count"].count().sort_values("count", ascending=False).head(10),
        x="artist", y="count", title="Music Activity By Artist", color_discrete_sequence=grayscale
    )

    # Music Activity By Track
    track_bar = px.bar(
        df_filtered.groupby("track", as_index=False)["count"].count().sort_values("count", ascending=False).head(10),
        x="track", y="count", title="Music Activity By Track", color_discrete_sequence=grayscale
    )

    # Music Activity By City And Genre
    city_genre_bar = px.bar(
        df_filtered.groupby(["city", "genre"], as_index=False)["count"].count(),
        x="genre", y="count", color="city", barmode="group",
        title="Music Activity By City And Genre", color_discrete_sequence=grayscale
    )

    # Music Activity By City And Artist
    city_artist_bar = px.bar(
        df_filtered.groupby(["city", "artist"], as_index=False)["count"].count().sort_values("count", ascending=False).head(20),
        x="artist", y="count", color="city", barmode="group",
        title="Music Activity By City And Artist", color_discrete_sequence=grayscale
    )

    # Music Activity By City And Track
    city_track_bar = px.bar(
        df_filtered.groupby(["city", "track"], as_index=False)["count"].count().sort_values("count", ascending=False).head(20),
        x="track", y="count", color="city", barmode="group",
        title="Music Activity By City And Track", color_discrete_sequence=grayscale
    )

    # Music Activity By City And DateTime
    time_grouped = (
        df_filtered.groupby(["city", "day", "time"], as_index=False)["count"].count()
        .sort_values(by=["day", "time"])
    )
    city_time_line = px.line(
        time_grouped,
        x="time", y="count", color="city", facet_col="day",
        title="Music Activity By City And DateTime", color_discrete_sequence=grayscale
    )

    return city_pie, genre_bar, artist_bar, track_bar, city_genre_bar, city_artist_bar, city_track_bar, city_time_line

if __name__ == '__main__':
    # app.run_server(host='0.0.0.0', port=3000)
    app.run(port=3000, debug=True, jupyter_mode='inline')      # embedded in cell